# 06. Machine Learning Model Development & Benchmarking
### Customer Behaviour Analysis & Churn Prediction System

We train and benchmark 4 classification algorithms using Stratified 5-Fold Cross Validation:
1. **Logistic Regression** (Interpretable baseline, probabilistic).
2. **Decision Tree** (Rule-based tree with pruning).
3. **Random Forest** (Ensemble of decorrelated trees).
4. **K-Nearest Neighbors (KNN)** (Instance-based similarity).

*(Note: In strict compliance with project guidelines, no XGBoost or SHAP is used).*


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("..")
from src.preprocessing import load_raw_data, prepare_full_pipeline
from src.evaluation import compare_models, cross_validate_model
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

data_path = os.path.join("..", "dataset", "customer_churn.csv")
raw_df = load_raw_data(data_path)
X, y, preprocessor, _ = prepare_full_pipeline(raw_df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=8, class_weight='balanced', random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights='distance')
}

trained_pipelines = {}
for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe

df_comp = compare_models(trained_pipelines, X_test, y_test)
df_comp


### Model Comparison Insights:
- **Random Forest** achieved the best balance between discrimination (**ROC-AUC 0.8435**) and recall (**77.5%**), while significantly reducing False Positives compared to single Decision Trees.
- **KNN** struggled on recall (51.9%) due to high-dimensional categorical sparsity.
